# E03 · Advanced Lane Finding V2.0

**基于传统计算机视觉的道路车道线检测与分析**  
MVAI Learning Center · 智能感知与自动驾驶技术

本 Notebook 以 CarND-Advanced-Lane-Lines 的教学路线为基础，组织五个连续实验：相机标定、颜色与梯度、透视变换、滑动窗口与曲线拟合、曲率/偏移与视频处理。

**运行要求：** Python + OpenCV + NumPy + Matplotlib；CPU 即可。本地 Jupyter、VS Code Notebook 和 Kaggle 均可运行。视频默认每段处理前 8 秒、约 10 FPS；可切换完整视频模式。

In [ ]:
from pathlib import Path
import sys, json, subprocess, shutil, math
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Video, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "data/e03/camera_cal").is_dir():
    raise FileNotFoundError("请先解压 E03 V2.0 完整实验包，并从实验包根目录打开 Notebook。")
sys.path.insert(0, str(ROOT))
from e03_lane_finding import (
    calibrate_camera, undistort, threshold_lane_pixels, perspective_matrices,
    sliding_window_search, search_around_poly, fit_lane_pixels,
    curvature_and_offset, LaneTracker
)
DATA=ROOT/"data/e03"; CAL=DATA/"camera_cal"; TEST=DATA/"test_images"; VIDEOS=DATA/"videos"; OUT=ROOT/"outputs"
for d in ["calibration","threshold","perspective","lane_fitting","images","videos"]:
    (OUT/d).mkdir(parents=True,exist_ok=True)

def show(items, cols=2, figsize=(14,8)):
    rows=math.ceil(len(items)/cols); fig,axes=plt.subplots(rows,cols,figsize=figsize,squeeze=False)
    for ax,(title,image) in zip(axes.flat,items):
        if image.ndim==2: ax.imshow(image,cmap="gray")
        else: ax.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        ax.set_title(title); ax.axis("off")
    for ax in axes.flat[len(items):]: ax.axis("off")
    plt.tight_layout(); plt.show()

def read_image(name):
    p=TEST/name
    im=cv2.imread(str(p))
    if im is None: raise FileNotFoundError(p)
    return im

print("OpenCV:",cv2.__version__)
print("标定图片:",len(list(CAL.glob("calibration*.jpg"))),"道路测试图片:",len(list(TEST.glob("*.jpg"))))
print("视频:",[p.name for p in sorted(VIDEOS.glob("*.mp4"))])

## 实验 E03-01 · 相机标定与图像畸变校正

**目标：** 使用 20 张棋盘格图片，检测 9×6 内角点，估计相机内参与畸变系数。

In [ ]:
calibration=calibrate_camera(sorted(CAL.glob("calibration*.jpg")),board_size=(9,6))
np.savez_compressed(OUT/"calibration/calibration_params.npz",mtx=calibration["mtx"],dist=calibration["dist"],rms=calibration["rms"],views=calibration["views"],image_size=np.array(calibration["image_size"]))
print(f"角点检测成功：{calibration['views']} / 20")
print(f"重投影 RMS：{calibration['rms']:.4f} px")
print("未检测成功：",[x[0] for x in calibration["failed"]])
im=cv2.imread(str(CAL/calibration["detected"][0]))
if im.shape[:2]==(721,1281): im=im[:720,:1280]
found,corners=cv2.findChessboardCornersSB(cv2.cvtColor(im,cv2.COLOR_BGR2GRAY),(9,6))
if found: cv2.drawChessboardCorners(im,(9,6),corners,found)
show([("9×6 棋盘格角点检测",im)])

### 探索任务 1 · 标定图片数量

比较使用 8、12 和全部成功标定视图时的 RMS。思考：图片数量越多是否必然越好？

In [ ]:
rows=[]
for n in (8,12,len(calibration["detected"])):
    r=calibrate_camera([CAL/x for x in calibration["detected"][:n]])
    rows.append((n,r["views"],r["rms"]))
for row in rows: print("输入视图 / 成功视图 / RMS(px):",row)

### 探索任务 2 · 角点检测方法

对 8 张代表图片比较传统角点检测与 SB 检测成功率。

In [ ]:
classic=[]; sb=[]
names=["calibration1.jpg","calibration4.jpg","calibration5.jpg","calibration8.jpg","calibration10.jpg","calibration12.jpg","calibration16.jpg","calibration20.jpg"]
for name in names:
    im=cv2.imread(str(CAL/name))
    if im.shape[:2]==(721,1281): im=im[:720,:1280]
    gray=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    classic.append(cv2.findChessboardCorners(gray,(9,6))[0])
    sb.append(cv2.findChessboardCornersSB(gray,(9,6))[0])
print("传统方法:",sum(classic),"/8; SB:",sum(sb),"/8")

### 探索任务 3 · 畸变校正参数 alpha

alpha 控制保留视野与边缘黑区之间的权衡。比较 0、0.5、1.0。

In [ ]:
road=read_image("test3.jpg")
show([(f"alpha={a}",undistort(road,calibration,alpha=a)) for a in (0,.5,1)],cols=3,figsize=(15,4))

## 实验 E03-02 · 颜色空间与梯度阈值提取

**目标：** 从道路背景中筛选车道线候选像素。

In [ ]:
image=read_image("test3.jpg"); stages=threshold_lane_pixels(image)
show([("原图",image),("H",stages["h"]),("L",stages["l"]),("S",stages["s"]),("黄色掩膜",stages["yellow_mask"]),("白色掩膜",stages["white_mask"]),("Sobel X",stages["sobel_x"]),("梯度掩膜",stages["gradient_mask"]),("融合二值图",stages["binary"])],cols=3,figsize=(15,12))
cv2.imwrite(str(OUT/"threshold/combined_binary_test3.png"),stages["binary"])

### 探索任务 4 · 黄色车道线 HLS 阈值

修改 H 通道下界，观察黄色标线提取的变化。

In [ ]:
h,l,s=cv2.split(cv2.cvtColor(image,cv2.COLOR_BGR2HLS))
panels=[("原图",image)]
for low in (5,12,20):
    mask=((h>=low)&(h<=38)&(s>=70)&(l>=65)).astype(np.uint8)*255
    panels.append((f"H lower={low}",mask))
show(panels,cols=2)

### 探索任务 5 · Sobel X 梯度阈值

比较不同梯度下界对车道边缘与背景纹理的影响。

In [ ]:
gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
sx=np.abs(cv2.Sobel(gray,cv2.CV_64F,1,0,ksize=3))
scaled=np.uint8(np.clip(sx*255/max(float(sx.max()),1),0,255))
show([("原图",image)]+[(f"Sobel X ≥ {t}",((scaled>=t)&(scaled<=180)).astype(np.uint8)*255) for t in (10,25,50)],cols=2)

### 探索任务 6 · AND / OR 融合策略

比较颜色掩膜与梯度掩膜的 AND、OR 组合。

In [ ]:
color=(stages["yellow_mask"]>0)|(stages["white_mask"]>0); grad=stages["gradient_mask"]>0
show([("Color",color.astype(np.uint8)*255),("Gradient",stages["gradient_mask"]),("AND",(color&grad).astype(np.uint8)*255),("OR",(color|grad).astype(np.uint8)*255),("教学融合结果",stages["binary"])],cols=2)

### 探索任务 7 · 跨场景阈值泛化

使用相同参数观察直道、弯道和复杂弯道的候选像素。

In [ ]:
scene_frames=[]
for name,t in [("solidWhiteRight.mp4",2),("project_video.mp4",2),("harder_challenge_video.mp4",15)]:
    cap=cv2.VideoCapture(str(VIDEOS/name)); cap.set(cv2.CAP_PROP_POS_MSEC,t*1000); ok,frame=cap.read(); cap.release()
    if not ok: raise RuntimeError("视频帧读取失败: "+name)
    scene_frames.append((name,frame))
show([(n+" · 原图",f) for n,f in scene_frames],cols=3,figsize=(15,4))
show([(n+" · 二值候选",threshold_lane_pixels(f)["binary"]) for n,f in scene_frames],cols=3,figsize=(15,4))

## 实验 E03-03 · 透视变换与道路俯视图

**目标：** 使用单应性矩阵将驾驶视角转换为近似俯视视角。初始坐标来自原项目 Writeup Template，适用于本组 1280×720 参考画面。

In [ ]:
M,Minv,src,dst=perspective_matrices(image.shape)
marked=image.copy(); cv2.polylines(marked,[src.astype(np.int32)],True,(0,0,255),4)
bird=cv2.warpPerspective(stages["binary"],M,(image.shape[1],image.shape[0]),flags=cv2.INTER_NEAREST)
show([("源点",marked),("俯视二值图",bird)],cols=2)

### 探索任务 8 · 改变源点顶部位置

将源点顶部 y 坐标分别设置为 430、460、490，观察俯视图变化。

In [ ]:
panels=[("原始俯视图",bird)]
for top in (430,460,490):
    sp=np.float32([[585,top],[203,719],[1127,719],[695,top]])
    mm=cv2.getPerspectiveTransform(sp,dst)
    panels.append((f"source top y={top}",cv2.warpPerspective(stages["binary"],mm,(image.shape[1],image.shape[0]))))
show(panels,cols=2)

### 探索任务 9 · 改变目标区域宽度

比较目标区域宽度变化对俯视车道间距的影响。

In [ ]:
panels=[("原始目标区域",bird)]
for left,right in ((280,1000),(320,960),(360,920)):
    dp=np.float32([[left,0],[left,719],[right,719],[right,0]])
    mm=cv2.getPerspectiveTransform(src,dp)
    panels.append((f"target width={right-left}",cv2.warpPerspective(stages["binary"],mm,(image.shape[1],image.shape[0]))))
show(panels,cols=2)

### 探索任务 10 · 同一组坐标用于不同视频

比较三段视频的俯视图。讨论摄像机视角、道路宽度和安装位置变化为何会影响变换效果。

In [ ]:
show([(n+" · Bird's-eye",cv2.warpPerspective(threshold_lane_pixels(f)["binary"],*perspective_matrices(f.shape)[:1],(f.shape[1],f.shape[0]))) for n,f in scene_frames],cols=3,figsize=(15,4))

## 实验 E03-04 · 车道线像素搜索与二次曲线拟合

**目标：** 使用直方图、滑动窗口和二次多项式搜索左右车道线。模型：\(x(y)=Ay^2+By+C\)。

In [ ]:
search=sliding_window_search(bird,nwindows=9,margin=100,minpix=50)
fit=fit_lane_pixels(search,bird.shape)
print("左/右候选像素:",fit["left_count"],fit["right_count"])
print("几何检查:",fit["valid"],fit["reason"])
vis=cv2.cvtColor(bird,cv2.COLOR_GRAY2BGR)
for a,y0,b,y1,c,d in search["windows"]:
    cv2.rectangle(vis,(max(0,a),y0),(min(vis.shape[1]-1,b),y1),(0,255,0),2)
    cv2.rectangle(vis,(max(0,c),y0),(min(vis.shape[1]-1,d),y1),(255,0,0),2)
if fit["valid"]:
    cv2.polylines(vis,[np.column_stack((fit["left_fitx"],fit["ploty"])).astype(np.int32)],False,(0,255,255),4)
    cv2.polylines(vis,[np.column_stack((fit["right_fitx"],fit["ploty"])).astype(np.int32)],False,(0,255,255),4)
show([("滑动窗口与二次曲线拟合",vis)])

### 探索任务 11 · 滑动窗口数量

比较 6、9、12 个窗口时的左右像素数量和几何检查。

In [ ]:
for n in (6,9,12):
    f=fit_lane_pixels(sliding_window_search(bird,nwindows=n),bird.shape)
    print(n,"windows |",f["left_count"],f["right_count"],"|",f["valid"])

### 探索任务 12 · 搜索窗口宽度

比较 margin=60、100、140 对候选像素搜索的影响。

In [ ]:
for margin in (60,100,140):
    f=fit_lane_pixels(sliding_window_search(bird,margin=margin),bird.shape)
    print("margin",margin,"|",f["left_count"],f["right_count"],"|",f["reason"])

### 探索任务 13 · 最少像素数

改变 minpix，观察窗口中心是否更容易被少量噪声带偏。

In [ ]:
for minpix in (20,50,100):
    f=fit_lane_pixels(sliding_window_search(bird,minpix=minpix),bird.shape)
    print("minpix",minpix,"|",f["left_count"],f["right_count"],"|",f["valid"])

### 探索任务 14 · 全图搜索与上一帧曲线附近搜索

比较完整滑动窗口搜索与使用前一帧曲线缩小搜索范围的思路。

In [ ]:
if fit["valid"]:
    local=search_around_poly(bird,fit["left_fit"],fit["right_fit"],margin=90)
    print("全图搜索:",len(search["leftx"]),len(search["rightx"]))
    print("曲线附近搜索:",len(local["leftx"]),len(local["rightx"]))
else:
    print("当前测试图拟合未通过几何检查，可更换 test1.jpg 或 test2.jpg 重试。")

## 实验 E03-05 · 曲率、车辆偏移与连续视频检测

**目标：** 将前四个实验连接为完整流程。米制计算使用车道宽度 3.7 m、纵向观察范围 30 m 的教学假设，不代表真实测量。

In [ ]:
tracker=LaneTracker(use_ema=False)
result=tracker.process(image,calibration)
show([("原始道路图像",image),("车道线检测结果",result["output"])])
if result["valid"]:
    print("估计曲率半径:",result["metrics"]["mean_curvature_m"],"m")
    print("车辆横向偏移:",result["metrics"]["vehicle_offset_m"],"m")
else: print("本帧检测不可靠:",result["reason"])

### 探索任务 15 · 改变车道宽度假设

观察车道宽度假设 3.3、3.7、4.0 米对曲率和偏移数值的影响。

In [ ]:
if result["valid"]:
    f={k:result[k] for k in ("left_fit","right_fit","left_fitx","right_fitx","ploty")}
    for width in (3.3,3.7,4.0):
        m=curvature_and_offset(f,result["warped"].shape,lane_width_m=width,road_length_m=30)
        print(width,"m → curvature",m["mean_curvature_m"],"m; offset",m["vehicle_offset_m"],"m")
else: print("当前帧无可靠拟合，请检查上面的候选像素图。")

### 探索任务 16 · 曲线拟合系数与曲率

尝试改变左右二次曲线的 A 系数，观察曲率如何变化，并解释二次项对弯曲程度的影响。

In [ ]:
if result["valid"]:
    base={k:(v.copy() if isinstance(v,np.ndarray) else v) for k,v in {k:result[k] for k in ("left_fit","right_fit","left_fitx","right_fitx","ploty")}.items()}
    for factor in (.9,1.,1.1):
        alt={k:(v.copy() if isinstance(v,np.ndarray) else v) for k,v in base.items()}
        alt["left_fit"][0]*=factor;alt["right_fit"][0]*=factor
        alt["left_fitx"]=np.polyval(alt["left_fit"],alt["ploty"]);alt["right_fitx"]=np.polyval(alt["right_fit"],alt["ploty"])
        print("A ×",factor,"→",curvature_and_offset(alt,result["warped"].shape)["mean_curvature_m"],"m")

### 探索任务 17 · EMA 平滑参数

比较不同 EMA 参数对连续检测曲线的平滑程度。EMA 只能缓解部分抖动，不能修复错误检测。

In [ ]:
def sample_bottom_positions(alpha,seconds=4,target_fps=5):
    cap=cv2.VideoCapture(str(VIDEOS/"harder_challenge_video.mp4"));fps=cap.get(cv2.CAP_PROP_FPS) or 25
    tracker=LaneTracker(use_ema=True,ema_alpha=alpha);xs=[];idx=0;step=max(1,round(fps/target_fps))
    while idx<seconds*fps:
        ok,frame=cap.read()
        if not ok:break
        if idx%step==0:
            r=tracker.process(frame,calibration)
            if r["valid"]:xs.append(float((r["left_fitx"][-1]+r["right_fitx"][-1])/2))
        idx+=1
    cap.release();return xs
for alpha in (.15,.25,.5):
    xs=sample_bottom_positions(alpha)
    print("EMA alpha",alpha,"有效采样点",len(xs),"位置标准差(px)",round(float(np.std(xs)),2) if xs else "无有效结果")

### 探索任务 18 · 三段视频比较与失败分析

分别运行三段视频，观察直道、普通弯道和复杂弯道的输出。

In [ ]:
VIDEO_MODE="short"  # 改为 "full" 处理完整视频
SHORT_CLIP_SECONDS=8
PROCESS_FPS=10 if VIDEO_MODE=="short" else 5
names=["solidWhiteRight.mp4","project_video.mp4","harder_challenge_video.mp4"]
for name in names:
    cmd=[sys.executable,str(ROOT/"process_e03_video.py"),"--video",str(VIDEOS/name),"--output-dir",str(OUT/"videos"),"--calibration",str(OUT/"calibration/calibration_params.npz"),"--max-seconds","0" if VIDEO_MODE=="full" else str(SHORT_CLIP_SECONDS),"--process-fps",str(PROCESS_FPS)]
    subprocess.run(cmd,cwd=str(ROOT),check=True)
    summary=json.loads((OUT/"videos"/(Path(name).stem+"_summary.json")).read_text())
    print(name,"|",summary["processed_duration_s"],"s | 几何检查覆盖率",summary["valid_frame_coverage_percent"],"%（不是准确率）")
    display(Video(str(summary["compare_path"]),embed=False,html_attributes="controls muted playsinline"))

## 实验总结与课后思考探索

### 复杂道路为什么检测失败？

在 `harder_challenge_video.mp4` 中，算法可能出现车道边界漂移、误选或局部拟合不合理。这些现象是本实验保留的真实结果，不要求本次实验将其全部修复。

请选择一个失败片段，记录时间点，并回答：

1. 失败来自颜色/梯度候选像素不足，还是候选像素混入阴影、护栏或路面接缝？
2. 透视变换源点或目标点不合适，会怎样影响俯视图中的车道宽度和曲线形状？
3. 滑动窗口为何会从真实车道线跳到其他高对比度结构？可以增加什么几何约束？
4. 二次多项式是否足够？你会优先改进候选像素筛选、拟合模型还是时序策略？
5. 自适应阈值、左右车道宽度约束或跨帧状态判断可能改善什么，又会带来什么新问题？

**提交建议：** 截取至少一张失败帧，附原图、二值图、俯视图和拟合结果，提出一种可验证的改进思路即可，不要求完成训练或实现完整算法。

**边界说明：** 本实验是传统计算机视觉教学案例，不是自动驾驶安全系统。曲率与车辆偏移基于固定尺度假设；没有人工标注 Ground Truth，不能将几何检查通过率解释为识别准确率。